FASE 2 — Data Understanding: CONAPO
2.1 Primera celda Markdown

# Fase 2 — Data Understanding
## Construcción y validación de la base CONAPO

### Proyecto
Modelo econométrico de atractividad comercial municipal en México.

### Objetivo de esta etapa

Explorar y validar la información municipal de población e indicadores demográficos de CONAPO antes de integrarla con DENUE.

Para el proyecto se requieren principalmente:

- población municipal a mitad de año en 2020;
- población municipal a mitad de año en 2025;
- razón de dependencia demográfica en 2020;
- clave y nombre del municipio.

Estas variables permitirán construir posteriormente las densidades comerciales de 2020 y 2025 y aportar una de las variables explicativas del modelo econométrico.

En esta etapa todavía no se integrará CONAPO con DENUE ni se realizarán regresiones.

In [1]:
############### 2.2 Librerías y rutas

from pathlib import Path
import zipfile

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [2]:
#####2.2 Verificar la ruta de CONAPO

# Ruta raíz del proyecto
PROJECT_ROOT = Path.cwd().parent

# Carpeta de datos CONAPO
CONAPO_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "conapo"
)

# Archivo original
archivo_conapo = (
    CONAPO_DIR
    / "00_Republica_mexicana.zip"
)

print("Ruta del proyecto:")
print(PROJECT_ROOT)

print("\nRuta del archivo CONAPO:")
print(archivo_conapo)

print("\n¿Existe el archivo?:", archivo_conapo.exists())

Ruta del proyecto:
c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico

Ruta del archivo CONAPO:
c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico\data\raw\conapo\00_Republica_mexicana.zip

¿Existe el archivo?: True


2.3 Revisar qué contiene el ZIP

### 2.3 Inspección del paquete CONAPO

Antes de cargar la información se revisa la estructura interna del archivo comprimido.

El objetivo es identificar las bases disponibles y seleccionar el archivo que contiene la población municipal y los indicadores demográficos necesarios para el proyecto.

In [3]:
with zipfile.ZipFile(archivo_conapo, "r") as z:

    contenido_conapo = z.namelist()

    print(
        f"Número de archivos encontrados: "
        f"{len(contenido_conapo)}\n"
    )

    for nombre in contenido_conapo:
        print(nombre)

Número de archivos encontrados: 4

00_Republica_mexicana/1_Grupo_Quinq_00_RM.xlsx
00_Republica_mexicana/2_Gran_Gedad_00_RM.xlsx
00_Republica_mexicana/3_Indicadores_Dem_00_RM.xlsx
00_Republica_mexicana/4_Descriptor_BD_00_RM.xlsx


### 2.4 Inspección de la base de indicadores demográficos

Se identifica y revisa el archivo de indicadores demográficos municipales incluido en el paquete de CONAPO.

Esta base será utilizada para obtener:

- población municipal en 2020;
- población municipal en 2025;
- razón de dependencia demográfica en 2020;
- claves y nombres municipales.

Antes de realizar transformaciones se revisarán sus hojas, dimensiones, columnas y una muestra de registros.

In [4]:
from io import BytesIO

with zipfile.ZipFile(archivo_conapo, "r") as z:

    # Identificar automáticamente el archivo de indicadores
    archivos_indicadores = [
        nombre for nombre in z.namelist()
        if nombre.endswith("3_Indicadores_Dem_00_RM.xlsx")
    ]

    print("Archivos encontrados:")
    for nombre in archivos_indicadores:
        print(nombre)

    archivo_indicadores_zip = archivos_indicadores[0]

    # Leer el Excel desde memoria
    datos_excel = BytesIO(
        z.read(archivo_indicadores_zip)
    )

    excel_conapo = pd.ExcelFile(datos_excel)

    print("\nHojas disponibles:")
    print(excel_conapo.sheet_names)

Archivos encontrados:
00_Republica_mexicana/3_Indicadores_Dem_00_RM.xlsx

Hojas disponibles:
['Sheet 1']


In [5]:
############2.5 Leer una muestra

# Leer las primeras 10 filas de la primera hoja
muestra_conapo = pd.read_excel(
    excel_conapo,
    sheet_name=excel_conapo.sheet_names[0],
    nrows=10
)

print("Dimensiones de la muestra:")
print(muestra_conapo.shape)

print("\nNúmero de columnas:")
print(muestra_conapo.shape[1])

print("\nColumnas:")
print(muestra_conapo.columns.tolist())

display(muestra_conapo)

Dimensiones de la muestra:
(10, 29)

Número de columnas:
29

Columnas:
['CLAVE', 'CLAVE_ENT', 'NOM_ENT', 'NOM_MUN', 'AÑO', 'HOM_MIT_AÑO', 'MUJ_MIT_AÑO', 'POB_MIT_MUN', 'MUJ_00_14', 'HOM_00_14', 'POB_00_14', 'MUJ_15_64', 'HOM_15_64', 'POB_15_64', 'MUJ_60_MAS', 'HOM_60_MAS', 'POB_60_MAS', 'MUJ_65_MAS', 'HOM_65_MAS', 'POB_65_MAS', 'POB_MIT_ENT', 'EDAD_MED', 'POR_MUN', 'IND_ENV_60', 'IND_ENV_65', 'RHM', 'RAZ_DEP_ADU', 'RAZ_DEP_INF', 'RAZ_DEP']


,CLAVE,CLAVE_ENT,NOM_ENT,NOM_MUN,AÑO,HOM_MIT_AÑO,MUJ_MIT_AÑO,POB_MIT_MUN,MUJ_00_14,HOM_00_14,POB_00_14,MUJ_15_64,HOM_15_64,POB_15_64,MUJ_60_MAS,HOM_60_MAS,POB_60_MAS,MUJ_65_MAS,HOM_65_MAS,POB_65_MAS,POB_MIT_ENT,EDAD_MED,POR_MUN,IND_ENV_60,IND_ENV_65,RHM,RAZ_DEP_ADU,RAZ_DEP_INF,RAZ_DEP
0,1001,1,Aguascalientes,Aguascalientes,1990,243891,255948,499839,96929,99544,196473,147114,135446,282560,17066,13107,30173,11905,8901,20806,750856,20,66.57,15.36,10.59,95.29,7.36,69.53,76.90
1,1001,1,Aguascalientes,Aguascalientes,1991,252446,264829,517275,99105,101832,200937,153421,141454,294875,17632,13496,31128,12303,9160,21463,774794,20,66.76,15.49,10.68,95.32,7.28,68.14,75.42
2,1001,1,Aguascalientes,Aguascalientes,1992,261132,273812,534944,101353,104204,205557,159735,147502,307237,18216,13887,32103,12724,9426,22150,798983,20,66.95,15.62,10.78,95.37,7.21,66.91,74.11
3,1001,1,Aguascalientes,Aguascalientes,1993,269901,282863,552764,103642,106617,210259,166081,153581,319662,18791,14284,33075,13140,9703,22843,823342,20,67.14,15.73,10.86,95.42,7.15,65.78,72.92
4,1001,1,Aguascalientes,Aguascalientes,1994,278717,291960,570677,105888,108998,214886,172502,159730,332232,19378,14681,34059,13570,9989,23559,847751,21,67.32,15.85,10.96,95.46,7.09,64.68,71.77
5,1001,1,Aguascalientes,Aguascalientes,1995,287450,301057,588507,108108,111411,219519,178930,165719,344649,19988,15145,35133,14019,10320,24339,871727,21,67.51,16.00,11.09,95.48,7.06,63.69,70.76
6,1001,1,Aguascalientes,Aguascalientes,1996,295258,308985,604243,109876,113299,223175,184645,171275,355920,20615,15703,36318,14464,10684,25148,893303,21,67.64,16.27,11.27,95.56,7.07,62.70,69.77
7,1001,1,Aguascalientes,Aguascalientes,1997,302058,316333,618391,111304,114723,226027,190121,176307,366428,21241,16240,37481,14908,11028,25936,912513,22,67.77,16.58,11.47,95.49,7.08,61.68,68.76
8,1001,1,Aguascalientes,Aguascalientes,1998,308536,323544,632080,112581,115983,228564,195589,181166,376755,21902,16803,38705,15374,11387,26761,931005,22,67.89,16.93,11.71,95.36,7.10,60.67,67.77
9,1001,1,Aguascalientes,Aguascalientes,1999,314667,330625,645292,113746,117080,230826,201004,185823,386827,22617,17403,40020,15875,11764,27639,948799,22,68.01,17.34,11.97,95.17,7.15,59.67,66.82


### 2.6 Validación de variables requeridas

Se verifica que la base de indicadores demográficos contenga todas las variables necesarias para el proyecto.

Las variables requeridas son:

- `CLAVE`: clave municipal;
- `CLAVE_ENT`: clave de entidad federativa;
- `NOM_ENT`: nombre de la entidad;
- `NOM_MUN`: nombre del municipio;
- `AÑO`: año de referencia;
- `POB_MIT_MUN`: población municipal a mitad de año;
- `RAZ_DEP`: razón de dependencia demográfica.

In [6]:
variables_conapo_requeridas = [
    "CLAVE",
    "CLAVE_ENT",
    "NOM_ENT",
    "NOM_MUN",
    "AÑO",
    "POB_MIT_MUN",
    "RAZ_DEP"
]

validacion_variables_conapo = pd.DataFrame({
    "variable": variables_conapo_requeridas,
    "disponible": [
        variable in muestra_conapo.columns
        for variable in variables_conapo_requeridas
    ]
})

display(validacion_variables_conapo)

print(
    "\nVariables faltantes:",
    [
        variable
        for variable in variables_conapo_requeridas
        if variable not in muestra_conapo.columns
    ]
)

,variable,disponible
0,CLAVE,True
1,CLAVE_ENT,True
2,NOM_ENT,True
3,NOM_MUN,True
4,AÑO,True
5,POB_MIT_MUN,True
6,RAZ_DEP,True



Variables faltantes: []


In [7]:
#################### 2.7 Cargar la base completa

conapo_raw = excel_conapo.parse(
    sheet_name=0
)

print("Dimensiones de la base completa:")
print(conapo_raw.shape)

print("\nAño mínimo:")
print(conapo_raw["AÑO"].min())

print("\nAño máximo:")
print(conapo_raw["AÑO"].max())

print("\nNúmero de años disponibles:")
print(conapo_raw["AÑO"].nunique())

print("\nNúmero total de registros:")
print(f"{len(conapo_raw):,}")

Dimensiones de la base completa:
(126225, 29)

Año mínimo:
1990

Año máximo:
2040

Número de años disponibles:
51

Número total de registros:
126,225


2.8 Revisar específicamente 2020 y 2025

### 2.8 Cobertura municipal para 2020 y 2025

Debido a que el proyecto utiliza 2020 y 2025 para construir la variación de la densidad comercial, se revisa por separado la cobertura municipal de ambos años.

Se validan:

- número de registros;
- número de claves municipales únicas;
- claves duplicadas;
- valores faltantes en población;
- valores faltantes en razón de dependencia.

In [8]:
resumen_conapo_anios = []

for anio in [2020, 2025]:

    df_anio = conapo_raw[
        conapo_raw["AÑO"] == anio
    ].copy()

    resumen_conapo_anios.append({
        "anio": anio,
        "registros": len(df_anio),
        "claves_unicas": df_anio["CLAVE"].nunique(),
        "claves_duplicadas": df_anio["CLAVE"].duplicated().sum(),
        "nulos_poblacion": df_anio["POB_MIT_MUN"].isna().sum(),
        "nulos_raz_dep": df_anio["RAZ_DEP"].isna().sum()
    })

resumen_conapo_anios = pd.DataFrame(
    resumen_conapo_anios
)

display(resumen_conapo_anios)

,anio,registros,claves_unicas,claves_duplicadas,nulos_poblacion,nulos_raz_dep
0,2020,2475,2475,0,0,0
1,2025,2475,2475,0,0,0


In [9]:
#################   2.9 Normalización preliminar de CVEGEO

conapo_raw["CVEGEO"] = (
    conapo_raw["CLAVE"]
    .astype("string")
    .str.strip()
    .str.zfill(5)
)

display(
    conapo_raw[
        [
            "CLAVE",
            "CVEGEO",
            "NOM_ENT",
            "NOM_MUN",
            "AÑO"
        ]
    ].head(10)
)

,CLAVE,CVEGEO,NOM_ENT,NOM_MUN,AÑO
0,1001,01001,Aguascalientes,Aguascalientes,1990
1,1001,01001,Aguascalientes,Aguascalientes,1991
2,1001,01001,Aguascalientes,Aguascalientes,1992
3,1001,01001,Aguascalientes,Aguascalientes,1993
4,1001,01001,Aguascalientes,Aguascalientes,1994
5,1001,01001,Aguascalientes,Aguascalientes,1995
6,1001,01001,Aguascalientes,Aguascalientes,1996
7,1001,01001,Aguascalientes,Aguascalientes,1997
8,1001,01001,Aguascalientes,Aguascalientes,1998
9,1001,01001,Aguascalientes,Aguascalientes,1999


### 2.10 Revisión del descriptor oficial de CONAPO

Se revisa la documentación incluida en el paquete oficial de CONAPO para confirmar la definición de las variables utilizadas en el proyecto.

En particular se documentarán:

- `CLAVE`: clave municipal;
- `AÑO`: año de referencia;
- `POB_MIT_MUN`: población municipal a mitad de año;
- `RAZ_DEP`: razón de dependencia demográfica.

Esta revisión permite fundamentar las variables directamente en la documentación de la fuente y no únicamente a partir de los nombres de las columnas.

In [10]:
########  Primero identifiquemos las hojas del descriptor

with zipfile.ZipFile(archivo_conapo, "r") as z:

    archivos_descriptor = [
        nombre for nombre in z.namelist()
        if nombre.endswith("4_Descriptor_BD_00_RM.xlsx")
    ]

    print("Archivo descriptor:")
    print(archivos_descriptor[0])

    datos_descriptor = BytesIO(
        z.read(archivos_descriptor[0])
    )

    excel_descriptor = pd.ExcelFile(
        datos_descriptor
    )

    print("\nHojas disponibles:")
    print(excel_descriptor.sheet_names)

Archivo descriptor:
00_Republica_mexicana/4_Descriptor_BD_00_RM.xlsx

Hojas disponibles:
['Índice', '1_Grupo_Quinq_00_RM', '2_Gran_Gedad_00_RM', '3_Indicadores_Dem_00_RM', 'Clasif_Mun', 'Clasif_EF']


In [11]:
for hoja in excel_descriptor.sheet_names:
    print(hoja)

Índice
1_Grupo_Quinq_00_RM
2_Gran_Gedad_00_RM
3_Indicadores_Dem_00_RM
Clasif_Mun
Clasif_EF


In [12]:
hoja_indicadores = [
    hoja for hoja in excel_descriptor.sheet_names
    if "Indicadores" in hoja
][0]

descriptor_indicadores = pd.read_excel(
    excel_descriptor,
    sheet_name=hoja_indicadores,
    header=None
)

print("Hoja seleccionada:")
print(hoja_indicadores)

print("\nDimensiones:")
print(descriptor_indicadores.shape)

display(
    descriptor_indicadores.head(40)
)

Hoja seleccionada:
3_Indicadores_Dem_00_RM

Dimensiones:
(38, 6)


,0,1,2,3,4,5
0,NaN,NaN,Reconstrucción y proyecciones de la población ...,NaN,NaN,NaN
1,NaN,NaN,República Mexicana,NaN,NaN,NaN
2,NaN,NaN,Descriptor de datos,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,Indicadores demográficos diversos,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,Abreviación variable,Variable,Descripción o categorías,Tipo
7,NaN,NaN,CLAVE,Clave única de municipio,Según clasificación de municipios,Numérico
8,NaN,NaN,CLAVE_ENT,Clave de entidad federativa,Según clave de entidades federativas,Numérico
9,NaN,NaN,NOM_ENT,Nombre de entidad federativa,Nombre de la entidad federativa,Caracter


### 2.11 Construcción del diccionario técnico CONAPO

A partir del descriptor oficial se construye un diccionario técnico con las variables utilizadas en el proyecto.

Se conservarán únicamente las definiciones necesarias para identificar el municipio, seleccionar los años de análisis, obtener la población municipal y construir la variable demográfica explicativa.

In [13]:
# Leer nuevamente la hoja utilizando como encabezado
# la fila donde comienzan los nombres reales de las columnas

descriptor_conapo_limpio = pd.read_excel(
    excel_descriptor,
    sheet_name="3_Indicadores_Dem_00_RM",
    header=6
)

# Eliminar columnas completamente vacías
descriptor_conapo_limpio = (
    descriptor_conapo_limpio
    .dropna(axis=1, how="all")
)

# Limpiar espacios en nombres de columnas
descriptor_conapo_limpio.columns = (
    descriptor_conapo_limpio.columns
    .astype(str)
    .str.strip()
)

print("Columnas del descriptor limpio:")
print(descriptor_conapo_limpio.columns.tolist())

print("\nDimensiones:")
print(descriptor_conapo_limpio.shape)

display(descriptor_conapo_limpio.head(10))

Columnas del descriptor limpio:
['Abreviación variable', 'Variable', 'Descripción o categorías', 'Tipo']

Dimensiones:
(31, 4)


,Abreviación variable,Variable,Descripción o categorías,Tipo
0,CLAVE,Clave única de municipio,Según clasificación de municipios,Numérico
1,CLAVE_ENT,Clave de entidad federativa,Según clave de entidades federativas,Numérico
2,NOM_ENT,Nombre de entidad federativa,Nombre de la entidad federativa,Caracter
3,NOM_MUN,Nombre de municipio,Nombre del municipio,Caracter
4,AÑO,Año,Año,Numérico
5,HOM_MIT_AÑO,Hombres a mitad de año,Hombres a mitad de año,Numérico
6,MUJ_MIT_AÑO,Mujeres a mitad de año,Mujeres a mitad de año,Numérico
7,POB_MIT_MUN,Población a mitad de año del municipio,Población a mitad de año del municipio,Numérico
8,MUJ_00_14,Mujeres de 0 a 14 años,Mujeres de 0 a 14 años,Numérico
9,HOM_00_14,Hombres de 0 a 14 años,Hombres de 0 a 14 años,Numérico


In [14]:
#### 2.12 Extraer las variables que utilizaremos

variables_descriptor_interes = [
    "CLAVE",
    "CLAVE_ENT",
    "NOM_ENT",
    "NOM_MUN",
    "AÑO",
    "POB_MIT_MUN",
    "RAZ_DEP"
]

diccionario_tecnico_conapo = (
    descriptor_conapo_limpio[
        descriptor_conapo_limpio[
            "Abreviación variable"
        ].isin(variables_descriptor_interes)
    ]
    .copy()
    .reset_index(drop=True)
)

display(diccionario_tecnico_conapo)

,Abreviación variable,Variable,Descripción o categorías,Tipo
0,CLAVE,Clave única de municipio,Según clasificación de municipios,Numérico
1,CLAVE_ENT,Clave de entidad federativa,Según clave de entidades federativas,Numérico
2,NOM_ENT,Nombre de entidad federativa,Nombre de la entidad federativa,Caracter
3,NOM_MUN,Nombre de municipio,Nombre del municipio,Caracter
4,AÑO,Año,Año,Numérico
5,POB_MIT_MUN,Población a mitad de año del municipio,Población a mitad de año del municipio,Numérico
6,RAZ_DEP,Razón de dependencia total,Es igual a la suma de la población de 0 a 14 a...,Numérico


In [15]:
variables_encontradas_conapo = set(
    diccionario_tecnico_conapo[
        "Abreviación variable"
    ]
)

variables_faltantes_descriptor = [
    variable
    for variable in variables_descriptor_interes
    if variable not in variables_encontradas_conapo
]

print(
    "Variables solicitadas:",
    len(variables_descriptor_interes)
)

print(
    "Variables encontradas:",
    len(variables_encontradas_conapo)
)

print(
    "Variables faltantes:",
    variables_faltantes_descriptor
)

Variables solicitadas: 7
Variables encontradas: 7
Variables faltantes: []


### 2.12 Conclusión de Data Understanding — CONAPO

La revisión de la base de indicadores demográficos de CONAPO permitió confirmar que la fuente contiene la información necesaria para el proyecto.

Principales resultados:

- La base contiene 126,225 registros y 29 variables.
- El periodo disponible comprende de 1990 a 2040, equivalente a 51 años.
- Para 2020 se identificaron 2,475 municipios.
- Para 2025 se identificaron 2,475 municipios.
- No existen claves municipales duplicadas en los años seleccionados.
- No se encontraron valores faltantes en `POB_MIT_MUN`.
- No se encontraron valores faltantes en `RAZ_DEP`.
- La clave municipal puede normalizarse correctamente al formato `CVEGEO` de cinco posiciones.
- Las siete variables requeridas fueron localizadas tanto en la base como en el descriptor oficial.

La información se considera adecuada para iniciar la preparación de la base municipal CONAPO.

La base procesada utilizará una sola fila por municipio. Los años 2020 y 2025 se transformarán en columnas, por lo que la estructura final del proyecto continuará siendo de corte transversal municipal y no de panel.

FASE 3 — Data Preparation: CONAPO

3.1 Extraer 2020

# Fase 3 — Data Preparation
## Construcción de la base municipal CONAPO

La base original contiene múltiples años por municipio. Para el modelo econométrico se seleccionan únicamente 2020 y 2025.

La información será transformada a formato ancho para conservar una sola observación por municipio.

Se construirán:

- `POB_2020`: población municipal a mitad de año en 2020.
- `POB_2025`: población municipal a mitad de año en 2025.
- `RAZ_DEP_2020`: razón de dependencia demográfica en 2020.

`RAZ_DEP_2020` será posteriormente la variable explicativa demográfica X1 del modelo.

In [16]:
conapo_2020 = (
    conapo_raw[
        conapo_raw["AÑO"] == 2020
    ][
        [
            "CVEGEO",
            "NOM_ENT",
            "NOM_MUN",
            "POB_MIT_MUN",
            "RAZ_DEP"
        ]
    ]
    .copy()
)

conapo_2020 = conapo_2020.rename(
    columns={
        "NOM_ENT": "entidad",
        "NOM_MUN": "municipio",
        "POB_MIT_MUN": "POB_2020",
        "RAZ_DEP": "RAZ_DEP_2020"
    }
)

print("Dimensiones CONAPO 2020:")
print(conapo_2020.shape)

display(conapo_2020.head(10))

Dimensiones CONAPO 2020:
(2475, 5)


,CVEGEO,entidad,municipio,POB_2020,RAZ_DEP_2020
30,01001,Aguascalientes,Aguascalientes,968960,48.82
81,01002,Aguascalientes,Asientos,52700,62.80
132,01003,Aguascalientes,Calvillo,59333,63.35
183,01004,Aguascalientes,Cosío,17369,60.13
234,01005,Aguascalientes,Jesús María,132642,54.20
285,01006,Aguascalientes,Pabellón de Arteaga,48495,57.22
336,01007,Aguascalientes,Rincón de Romos,58520,59.97
387,01008,Aguascalientes,San José de Gracia,9751,64.63
438,01009,Aguascalientes,Tepezalá,22951,62.26
489,01010,Aguascalientes,El Llano,21230,58.66


In [17]:
################  3.2 Extraer 2025
conapo_2025 = (
    conapo_raw[
        conapo_raw["AÑO"] == 2025
    ][
        [
            "CVEGEO",
            "NOM_ENT",
            "NOM_MUN",
            "POB_MIT_MUN"
        ]
    ]
    .copy()
)

conapo_2025 = conapo_2025.rename(
    columns={
        "NOM_ENT": "entidad_2025",
        "NOM_MUN": "municipio_2025",
        "POB_MIT_MUN": "POB_2025"
    }
)

print("Dimensiones CONAPO 2025:")
print(conapo_2025.shape)

display(conapo_2025.head(10))


Dimensiones CONAPO 2025:
(2475, 4)


,CVEGEO,entidad_2025,municipio_2025,POB_2025
35,01001,Aguascalientes,Aguascalientes,1029221
86,01002,Aguascalientes,Asientos,57713
137,01003,Aguascalientes,Calvillo,60886
188,01004,Aguascalientes,Cosío,18719
239,01005,Aguascalientes,Jesús María,141363
290,01006,Aguascalientes,Pabellón de Arteaga,49080
341,01007,Aguascalientes,Rincón de Romos,60808
392,01008,Aguascalientes,San José de Gracia,10083
443,01009,Aguascalientes,Tepezalá,23292
494,01010,Aguascalientes,El Llano,21824


In [18]:
##################3.3 Antes de unir: comprobar que son las mismas claves

claves_conapo_2020 = set(
    conapo_2020["CVEGEO"]
)

claves_conapo_2025 = set(
    conapo_2025["CVEGEO"]
)

comunes_conapo = (
    claves_conapo_2020
    & claves_conapo_2025
)

solo_conapo_2020 = (
    claves_conapo_2020
    - claves_conapo_2025
)

solo_conapo_2025 = (
    claves_conapo_2025
    - claves_conapo_2020
)

print(
    f"Municipios 2020: "
    f"{len(claves_conapo_2020):,}"
)

print(
    f"Municipios 2025: "
    f"{len(claves_conapo_2025):,}"
)

print(
    f"Municipios comunes: "
    f"{len(comunes_conapo):,}"
)

print(
    f"Solo en 2020: "
    f"{len(solo_conapo_2020):,}"
)

print(
    f"Solo en 2025: "
    f"{len(solo_conapo_2025):,}"
)

Municipios 2020: 2,475
Municipios 2025: 2,475
Municipios comunes: 2,475
Solo en 2020: 0
Solo en 2025: 0


In [19]:
#3.4 Integrar 2020 y 2025
conapo_municipal = (
    conapo_2020
    .merge(
        conapo_2025[
            [
                "CVEGEO",
                "POB_2025"
            ]
        ],
        on="CVEGEO",
        how="inner",
        validate="one_to_one"
    )
    [
        [
            "CVEGEO",
            "entidad",
            "municipio",
            "POB_2020",
            "POB_2025",
            "RAZ_DEP_2020"
        ]
    ]
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "Dimensiones de la base municipal CONAPO:"
)

print(conapo_municipal.shape)

display(conapo_municipal.head(10))

Dimensiones de la base municipal CONAPO:
(2475, 6)


,CVEGEO,entidad,municipio,POB_2020,POB_2025,RAZ_DEP_2020
0,01001,Aguascalientes,Aguascalientes,968960,1029221,48.82
1,01002,Aguascalientes,Asientos,52700,57713,62.80
2,01003,Aguascalientes,Calvillo,59333,60886,63.35
3,01004,Aguascalientes,Cosío,17369,18719,60.13
4,01005,Aguascalientes,Jesús María,132642,141363,54.20
5,01006,Aguascalientes,Pabellón de Arteaga,48495,49080,57.22
6,01007,Aguascalientes,Rincón de Romos,58520,60808,59.97
7,01008,Aguascalientes,San José de Gracia,9751,10083,64.63
8,01009,Aguascalientes,Tepezalá,22951,23292,62.26
9,01010,Aguascalientes,El Llano,21230,21824,58.66


In [20]:
######################3.5 Control de calidad

print("CONTROL DE CALIDAD — CONAPO MUNICIPAL")
print("=" * 55)

print(
    "Número de municipios:",
    f"{len(conapo_municipal):,}"
)

print(
    "CVEGEO únicos:",
    f"{conapo_municipal['CVEGEO'].nunique():,}"
)

print(
    "CVEGEO duplicados:",
    conapo_municipal["CVEGEO"]
    .duplicated()
    .sum()
)

print(
    "Valores nulos totales:",
    conapo_municipal
    .isna()
    .sum()
    .sum()
)

print(
    "POB_2020 <= 0:",
    (
        conapo_municipal["POB_2020"]
        <= 0
    ).sum()
)

print(
    "POB_2025 <= 0:",
    (
        conapo_municipal["POB_2025"]
        <= 0
    ).sum()
)

print(
    "RAZ_DEP_2020 <= 0:",
    (
        conapo_municipal["RAZ_DEP_2020"]
        <= 0
    ).sum()
)

CONTROL DE CALIDAD — CONAPO MUNICIPAL
Número de municipios: 2,475
CVEGEO únicos: 2,475
CVEGEO duplicados: 0
Valores nulos totales: 0
POB_2020 <= 0: 0
POB_2025 <= 0: 0
RAZ_DEP_2020 <= 0: 0


In [21]:
assert len(conapo_municipal) == 2475, \
    "El número de municipios no coincide."

assert (
    conapo_municipal["CVEGEO"].nunique()
    == 2475
), "Existen problemas de unicidad en CVEGEO."

assert (
    conapo_municipal["CVEGEO"]
    .duplicated()
    .sum()
    == 0
), "Existen CVEGEO duplicados."

assert (
    conapo_municipal
    .isna()
    .sum()
    .sum()
    == 0
), "Existen valores faltantes."

assert (
    conapo_municipal["POB_2020"]
    .gt(0)
    .all()
), "Existen poblaciones 2020 no positivas."

assert (
    conapo_municipal["POB_2025"]
    .gt(0)
    .all()
), "Existen poblaciones 2025 no positivas."

assert (
    conapo_municipal["RAZ_DEP_2020"]
    .gt(0)
    .all()
), "Existen razones de dependencia no positivas."

print(
    "Todas las validaciones fueron "
    "superadas correctamente."
)

Todas las validaciones fueron superadas correctamente.


### 3.6 Exportación de la base municipal CONAPO

Después de superar los controles de calidad, la base municipal de CONAPO se guarda como archivo procesado.

La base contiene una observación por municipio y las variables:

- `POB_2020`: población municipal a mitad de año en 2020.
- `POB_2025`: población municipal a mitad de año en 2025.
- `RAZ_DEP_2020`: razón de dependencia demográfica en 2020.

`POB_2020` y `POB_2025` se utilizarán posteriormente como denominadores para construir la densidad comercial municipal, mientras que `RAZ_DEP_2020` será la variable explicativa demográfica X1.

In [22]:
# Ruta de salida
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

archivo_salida_conapo = (
    PROCESSED_DIR / "conapo_municipal.csv"
)

conapo_municipal.to_csv(
    archivo_salida_conapo,
    index=False,
    encoding="utf-8-sig"
)

print("Base CONAPO guardada correctamente.")
print(f"Ruta: {archivo_salida_conapo}")

Base CONAPO guardada correctamente.
Ruta: c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico\data\processed\conapo_municipal.csv


In [23]:
conapo_verificacion = pd.read_csv(
    archivo_salida_conapo,
    dtype={"CVEGEO": "string"}
)

print("Dimensiones:")
print(conapo_verificacion.shape)

print("\nCVEGEO únicos:")
print(conapo_verificacion["CVEGEO"].nunique())

print("\nValores nulos:")
print(conapo_verificacion.isna().sum().sum())

display(conapo_verificacion.head(10))

Dimensiones:
(2475, 6)

CVEGEO únicos:
2475

Valores nulos:
0


,CVEGEO,entidad,municipio,POB_2020,POB_2025,RAZ_DEP_2020
0,01001,Aguascalientes,Aguascalientes,968960,1029221,48.82
1,01002,Aguascalientes,Asientos,52700,57713,62.80
2,01003,Aguascalientes,Calvillo,59333,60886,63.35
3,01004,Aguascalientes,Cosío,17369,18719,60.13
4,01005,Aguascalientes,Jesús María,132642,141363,54.20
5,01006,Aguascalientes,Pabellón de Arteaga,48495,49080,57.22
6,01007,Aguascalientes,Rincón de Romos,58520,60808,59.97
7,01008,Aguascalientes,San José de Gracia,9751,10083,64.63
8,01009,Aguascalientes,Tepezalá,22951,23292,62.26
9,01010,Aguascalientes,El Llano,21230,21824,58.66
